# Bake-off candidate: abdullahtarek/football_analysis (ATFA)

Runs the ATFA football analysis pipeline (YOLOv8 + ByteTrack + KMeans team-color
classifier) on the canonical bake-off clip. Outputs trajectories parquet +
annotated video.

**Inputs:** `data/bakeoff_clip.mp4`, `data/bakeoff_clip_corners.json` (both from the
soccer-vision repo).

**Outputs:** `data/bakeoff_outputs/atfa/trajectories.parquet`,
`data/bakeoff_outputs/atfa/annotated.mp4`.

Run on Colab Pro with T4 or L4 GPU.

In [ ]:
from google.colab import userdata
GH_PAT = userdata.get('GITHUB_PAT')  # Set in Colab Secrets (🔑 sidebar) once
!pip install -q "ultralytics>=8.2" "supervision>=0.22" "scikit-learn>=1.5"
!git clone https://github.com/abdullahtarek/football_analysis /content/atfa
import sys; sys.path.insert(0, "/content/atfa")
!pip install -q "git+https://${GH_PAT}@github.com/PatrickJReed/soccer-vision.git"
from pathlib import Path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
INPUT_CLIP = Path('/content/drive/MyDrive/soccer-vision/bakeoff_clip.mp4')
OUT = Path("/content/output/atfa")
OUT.mkdir(parents=True, exist_ok=True)
assert INPUT_CLIP.exists(), f"Place bakeoff_clip.mp4 at {INPUT_CLIP}"

In [ ]:
# ATFA's Tracker wraps YOLOv8 + supervision ByteTrack
# The exact module path is /content/atfa/trackers/tracker.py
from trackers.tracker import Tracker  # ATFA layout
import cv2

# Read all frames once (ATFA's Tracker.get_object_tracks expects a list of frames)
cap = cv2.VideoCapture(str(INPUT_CLIP))
fps = cap.get(cv2.CAP_PROP_FPS)
video_frames = []
while True:
    ok, frame = cap.read()
    if not ok:
        break
    video_frames.append(frame)
cap.release()
print(f"Loaded {len(video_frames)} frames at {fps} fps")

# TODO when running: confirm the right YOLOv8 weights path. ATFA README references
# 'models/best.pt' (a custom-trained model). For the bake-off, the default
# pretrained YOLOv8x weights file should work as a fallback.
tracker = Tracker("yolov8x.pt")
tracks = tracker.get_object_tracks(video_frames, read_from_stub=False)
# tracks is a dict with keys 'players', 'referees', 'ball' -> list-per-frame of {track_id: {"bbox": [x1, y1, x2, y2]}}

In [ ]:
# Sketch — actual code mirrors ATFA's TeamAssigner usage in
# /content/atfa/team_assigner/team_assigner.py
from team_assigner.team_assigner import TeamAssigner

team_assigner = TeamAssigner()
team_assigner.assign_team_color(video_frames[0], tracks["players"][0])

# Translate ATFA's per-frame dicts to schema rows
records: list[dict] = []
for frame_idx in range(len(video_frames)):
    for cls_name, cls_label in [("players", "player"), ("referees", "referee"), ("ball", "ball")]:
        for tid, info in tracks[cls_name][frame_idx].items():
            x1, y1, x2, y2 = info["bbox"]
            cx, cy = (x1 + x2) / 2, y2
            team = "unknown"
            if cls_name == "players":
                team_id = team_assigner.get_player_team(video_frames[frame_idx], info["bbox"], tid)
                team = {1: "own", 2: "opp"}.get(team_id, "unknown")
            elif cls_name == "referees":
                team = "ref"
            records.append({
                "frame": frame_idx,
                "t_seconds": frame_idx / fps,
                "track_id": int(tid),
                "x_px": cx,
                "y_px": cy,
                "bbox_x1": x1, "bbox_y1": y1, "bbox_x2": x2, "bbox_y2": y2,
                "class": cls_name[:-1] if cls_name.endswith("s") else cls_name,  # players->player, ball->ball
                "team": team,
                "conf": 1.0,  # ATFA doesn't expose per-detection confidence; fixed value
            })

In [ ]:
import pandas as pd
from soccer_vision.io.schema import validate_trajectories

df = pd.DataFrame(records)
df = df.astype({"frame": "int64", "track_id": "int64"})
validate_trajectories(df)
df.to_parquet(OUT / "trajectories.parquet")
print(f"Wrote {OUT / 'trajectories.parquet'}: {len(df)} rows")

In [ ]:
# Render boxes + IDs onto the clip
import supervision as sv

cap = cv2.VideoCapture(str(INPUT_CLIP))
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
writer = cv2.VideoWriter(str(OUT / "annotated.mp4"),
                         cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

for frame_idx, group in df.groupby("frame"):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()
    if not ok:
        break
    # TODO when running: render boxes + labels using cv2.rectangle / cv2.putText,
    # or use supervision annotators. Color by team: own=green, opp=red, ref=yellow.
    writer.write(frame)

writer.release()
cap.release()
print(f"Wrote {OUT / 'annotated.mp4'}")

In [ ]:
# Download to commit at: data/bakeoff_outputs/atfa/{trajectories.parquet,annotated.mp4}
from google.colab import files
files.download(str(OUT / "trajectories.parquet"))
files.download(str(OUT / "annotated.mp4"))